# 6.2 — Training & Perbandingan Model (Pixel-Level)

Pada versi sebelumnya model dilatih dari **rata-rata 6 band per poligon** (hanya ~2.658 baris),
padahal saat produksi peta (6.3/6.5/6.6) prediksi dilakukan **per piksel**. Selain itu 165 poligon
hilang karena nodata/awan dan validasi memakai split biasa sehingga risiko overfitting tidak terukur.

Perbaikan pada versi ini:

1. **Data latih pixel-level** — 30 titik sampel per poligon (±84 ribu titik) dibaca lewat openEO
   `aggregate_spatial`, dengan **masking awan SCL** (kelas 3, 8–11 dijadikan nodata sebelum
   komposit median). Poligon yang tadinya hilang ikut terwakili.
2. **Inkonsistensi latih–inferensi hilang** — fitur yang dilatih persis sama dengan fitur saat
   klasifikasi raster (6 band + NDVI, NDWI, NDBI, NDMI, EVI, SAVI).
3. **Validasi group-aware** — pembagian train/test *per poligon* (`StratifiedGroupKFold`,
   grup = id poligon), sehingga piksel dari poligon yang sama tidak bocor ke data uji.
   Metrik yang dilaporkan adalah akurasi + macro-F1 rata-rata silang 5-lipatan.

In [1]:
import glob
import os

import numpy as np
import pandas as pd

SEED = 42
PIX_CSV = "jatim/features/pixels_all.csv"
FEAT_DIR = "jatim/features"
MODEL_DIR = "jatim/model"
IMG_DIR = "jatim/gambar"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
NAMES = {1: "Sawah", 2: "Bangunan", 3: "Mangrove", 4: "Lahan hijau", 5: "Laut", 6: "Danau"}

if os.path.exists(PIX_CSV):
    df = pd.read_csv(PIX_CSV)
    SUMBER = "pixel"
else:  # fallback: komposit mean lama bila fase pixels belum dijalankan
    files = sorted(f for f in glob.glob(os.path.join(FEAT_DIR, "features_class[1-6].csv")))
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    SUMBER = "mean-lama"

print("sumber data:", SUMBER)
print("baris:", len(df), "| kolom:", list(df.columns))
print("NaN per band:", df[BANDS].isna().sum().to_dict())
print("poligon unik:", df["poligon"].nunique() if "poligon" in df.columns else "-")
print(df["class_id"].value_counts().sort_index().to_dict())

sumber data: pixel
baris: 84016 | kolom: ['pid', 'B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'poligon', 'class_id']
NaN per band: {'B02': 5, 'B03': 5, 'B04': 5, 'B08': 5, 'B11': 5, 'B12': 5}
poligon unik: 2821
{1: 15000, 2: 15000, 3: 14701, 4: 14870, 5: 15000, 6: 9445}


In [2]:
import numpy as np

# Fitur = 6 band + 6 indeks (SAMA dengan yang dipakai saat inferensi raster)
b = {c: pd.to_numeric(df[c], errors="coerce") for c in BANDS}
B02, B03, B04, B08, B11, B12 = (b[c] for c in BANDS)
df["NDVI"] = (B08 - B04) / (B08 + B04)
df["NDWI"] = (B03 - B08) / (B03 + B08)
df["NDBI"] = (B11 - B08) / (B11 + B08)
df["NDMI"] = (B08 - B11) / (B08 + B11)
df["EVI"] = 2.5 * (B08 - B04) / (B08 + 6 * B04 - 7.5 * B02 + 1)
df["SAVI"] = ((B08 - B04) / (B08 + B04 + 0.5)) * 1.5
FEATS = BANDS + ["NDVI", "NDWI", "NDBI", "NDMI", "EVI", "SAVI"]

# Pembulatan nol (piksel nodata) membuat penyebut 0 -> hasil bagi bisa +-inf.
# Ganti inf jadi NaN dulu supaya ikut terbuang saat dropna (model tolak inf).
df[FEATS] = df[FEATS].replace([np.inf, -np.inf], np.nan)

n_awal = len(df)
df = df.dropna(subset=FEATS).reset_index(drop=True)
print("fitur:", FEATS)
print(f"sampel bersih: {len(df)} (dibuang {n_awal - len(df)} baris NaN/0-denom)")
df.to_csv(os.path.join(MODEL_DIR, "jatim_dataset_pixel.csv"), index=False)
df.groupby("class_id")[FEATS].mean().round(3).to_string()

fitur: ['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'NDMI', 'EVI', 'SAVI']
sampel bersih: 84010 (dibuang 6 baris NaN/0-denom)


'              B02       B03       B04       B08       B11       B12   NDVI   NDWI   NDBI   NDMI    EVI   SAVI\nclass_id                                                                                                     \n1         560.281   825.726   733.945  2939.908  2238.289  1356.982  0.594 -0.552 -0.138  0.138  1.849  0.891\n2         803.118  1006.808  1041.597  2570.415  2432.779  1844.183  0.425 -0.433 -0.022  0.022  1.452  0.638\n3         413.740   645.705   473.955  2279.355   931.688   489.240  0.608 -0.509 -0.416  0.416  2.410  0.912\n4         584.017   811.367   713.463  2798.185  2115.576  1315.030  0.583 -0.537 -0.142  0.142  1.901  0.875\n5         564.723   471.269   387.203   352.925   357.910   317.321 -0.063  0.167  0.004 -0.004  0.101 -0.094\n6         487.801   706.773   561.859  2195.108  1544.803   914.276  0.522 -0.435 -0.147  0.147  2.305  0.783'

## Split train–test per poligon + validasi silang group-aware

`StratifiedGroupKFold(grup=id_poligon)` memastikan seluruh piksel satu poligon selalu berada di
sisi yang sama (train ATAU test) — syarat utama agar angka akurasi tidak bengkak karena kebocoran
spasial antar piksel tetangga.

In [3]:
import joblib
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgbmod import XGBShift   # kelas portabel (bukan __main__)



X = df[FEATS].to_numpy()
y = df["class_id"].to_numpy().astype(int)
groups = df["poligon"].to_numpy() if "poligon" in df.columns else np.arange(len(df))

# Split 80/20 BERBASIS GRUP: id poligon di-stratify lalu seluruh pikselnya ikut
# ke sisi yang sama, sehingga tidak ada piksel poligon yang bocor train<->test.
uniq = np.unique(groups)
first_y = pd.Series(y).groupby(groups).first().loc[uniq].to_numpy()
u_tr, u_te = train_test_split(uniq, test_size=0.2, random_state=SEED, stratify=first_y)
idx_tr = np.where(np.isin(groups, u_tr))[0]
idx_te = np.where(np.isin(groups, u_te))[0]
print(f"train: {len(idx_tr)} piksel dari {len(u_tr)} poligon")
print(f"test : {len(idx_te)} piksel dari {len(u_te)} poligon")
print("irisan poligon train-test:", len(set(u_tr) & set(u_te)))


train: 67106 piksel dari 2256 poligon
test : 16904 piksel dari 565 poligon
irisan poligon train-test: 0


In [4]:
# Kandidat model + hyperparameter kecil.
# CATATAN: paralel DIBATASI 2 thread/estimator -- n_jobs=-1 di mesin 4-core/8 GB
# pernah membuat proses mati OOM di tengah training. Hasil tiap varian disimpan
# inkremental, jadi bila terputus, run berikutnya resume (CV tidak diulang).
import gc

rng = np.random.RandomState(SEED)
svm_cap = 8000  # SVC RBF O(n^2) -> subsample stratified agar muat
svm_idx = np.concatenate([
    rng.choice(np.where(y[idx_tr] == c)[0],
               size=min(svm_cap // 6, int((y[idx_tr] == c).sum())), replace=False)
    for c in sorted(NAMES)
])
rng.shuffle(svm_idx)


def svm_xy():
    return X[idx_tr][svm_idx], y[idx_tr][svm_idx]


NJ = 2
MODELS = {
    "RandomForest": [
        lambda d: RandomForestClassifier(n_estimators=300, max_depth=25, min_samples_leaf=1,
                                         random_state=SEED, n_jobs=NJ),
        lambda d: RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=NJ),
    ],
    "ExtraTrees": [
        lambda d: ExtraTreesClassifier(n_estimators=300, max_depth=25, min_samples_leaf=1,
                                       random_state=SEED, n_jobs=NJ),
        lambda d: ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=NJ),
    ],
    "HistGradBoost": [
        lambda d: HistGradientBoostingClassifier(max_depth=12, random_state=SEED),
        lambda d: HistGradientBoostingClassifier(random_state=SEED),
        lambda d: HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05,
                                                 max_leaf_nodes=63, min_samples_leaf=30,
                                                 l2_regularization=1.0, random_state=SEED),
    ],
    "XGBoost": [
        lambda d: XGBShift(n_estimators=400, max_depth=8, learning_rate=0.1,
                           subsample=0.9, colsample_bytree=0.9,
                           tree_method="hist", eval_metric="mlogloss",
                           random_state=SEED, n_jobs=NJ),
        lambda d: XGBShift(n_estimators=500, max_depth=8, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                           tree_method="hist", eval_metric="mlogloss",
                           random_state=SEED, n_jobs=NJ),
    ],
    "SVM-RBF": [
        lambda d: make_pipeline(StandardScaler(), SVC(C=1, random_state=SEED)),
        lambda d: make_pipeline(StandardScaler(), SVC(C=10, random_state=SEED)),
    ],
    "LogReg": [
        lambda d: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000,
                                                                     random_state=SEED)),
    ],
}


def get_xy(name):
    return svm_xy() if name == "SVM-RBF" else (X[idx_tr], y[idx_tr])


cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

# ---- hasil INKREMENTAL: disimpan tiap varian, dipakai ulang kalau terputus ----
PART_CSV = os.path.join(MODEL_DIR, "jatim_model_compare_part.csv")
rows = []
if os.path.exists(PART_CSV):
    try:
        _d = pd.read_csv(PART_CSV).dropna(subset=["cv5_f1_macro"])
        rows = _d.to_dict("records")
        print(f"[resume] {len(rows)} varian hasil lama dipakai ulang (CV tidak diulang)")
    except Exception as _e:
        print("[resume] gagal membaca, mulai dari nol:", _e)
        rows = []

EVAL_FITTED = {}


def _simpan():
    pd.DataFrame(rows).to_csv(PART_CSV, index=False)


for name, grid in MODELS.items():
    Xa, ya = get_xy(name)
    ga = groups[idx_tr] if name != "SVM-RBF" else groups[idx_tr][svm_idx]

    for k, factory in enumerate(grid):
        if any(r["model"] == name and int(r["varian"]) == k for r in rows):
            continue                      # sudah dinilai pada run sebelumnya
        accs, f1s = [], []
        for tri, vai in cv.split(Xa, ya, ga):
            m = factory(None)
            m.fit(Xa[tri], ya[tri])
            pv = m.predict(Xa[vai])
            accs.append(accuracy_score(ya[vai], pv))
            f1s.append(f1_score(ya[vai], pv, average="macro", zero_division=0))
            del m, pv
        rec = {"model": name, "varian": k, "cv5_accuracy": float(np.mean(accs)),
               "cv5_accuracy_std": float(np.std(accs)), "cv5_f1_macro": float(np.mean(f1s)),
               "cv5_f1_std": float(np.std(f1s))}
        rows.append(rec)
        _simpan()                         # <-- tersimpan sebelum lanjut
        print(f"{name} var{k}: cv5_acc={rec['cv5_accuracy']:.4f}"
              f"+-{rec['cv5_accuracy_std']:.4f} f1={rec['cv5_f1_macro']:.4f}", flush=True)
        gc.collect()

    # varian terbaik utk model ini -> refit HANYA ke train, evaluasi sekali di test
    mine = [i for i, r in enumerate(rows) if r["model"] == name]
    bi = max(mine, key=lambda i: rows[i]["cv5_f1_macro"])
    need_test = pd.isna(rows[bi].get("test_accuracy", np.nan))
    m = grid[int(rows[bi]["varian"])](None)
    m.fit(*get_xy(name))                  # train saja -> evaluasi test tetap jujur
    EVAL_FITTED[name] = m
    pv = m.predict(X[idx_te])
    if need_test:
        rows[bi]["test_accuracy"] = accuracy_score(y[idx_te], pv)
        rows[bi]["test_f1_macro"] = f1_score(y[idx_te], pv, average="macro", zero_division=0)
        rows[bi]["best_varian"] = int(rows[bi]["varian"])
        _simpan()
    print(f"  -> varian #{int(rows[bi]['varian'])} | test acc="
          f"{rows[bi]['test_accuracy']:.4f} f1={rows[bi]['test_f1_macro']:.4f}", flush=True)
    del m, pv, Xa, ya
    gc.collect()

print(f"\nselesai CV: {len(rows)} baris dari {sum(len(g) for g in MODELS.values())} varian")


RandomForest var0: cv5_acc=0.6448+-0.0106 f1=0.6326


RandomForest var1: cv5_acc=0.6435+-0.0099 f1=0.6307


  -> varian #0 | test acc=0.6302 f1=0.6191


ExtraTrees var0: cv5_acc=0.6430+-0.0101 f1=0.6292


ExtraTrees var1: cv5_acc=0.6392+-0.0096 f1=0.6254


  -> varian #0 | test acc=0.6294 f1=0.6178


... (9 outputs left out)

  -> varian #1 | test acc=0.6416 f1=0.6307


LogReg var0: cv5_acc=0.6238+-0.0122 f1=0.6006


  -> varian #0 | test acc=0.6182 f1=0.5992



selesai CV: 12 baris dari 12 varian


In [5]:
res = pd.DataFrame(rows).sort_values("cv5_f1_macro", ascending=False).reset_index(drop=True)
res["accuracy_gap"] = res["cv5_accuracy"] - res.get("test_accuracy", np.nan)
print(res.round(4).to_string(index=False))
res.to_csv(os.path.join(MODEL_DIR, "jatim_model_compare.csv"), index=False)

# model final = kombinasi (model, varian) dengan CV-F1 tertinggi, refit ke SELURUH data
top = res.iloc[0]
name, var = top["model"], int(top["best_varian"])
final = MODELS[name][var](None)
if name == "SVM-RBF":
    Xf, yf = svm_xy()          # SVC RBF tetap pakai subsample agar waktu inferensi wajar
else:
    Xf, yf = X, y              # model berbasis pohon dilatih ke seluruh piksel train+test
final.fit(Xf, yf)
joblib.dump(final, os.path.join(MODEL_DIR, "jatim_best.joblib"))
print("\nMODEL TERBAIK:", name, "varian", var,
      "| cv5_f1 =", round(top["cv5_f1_macro"], 4),
      "| test acc =", round(top["test_accuracy"], 4))
print("Gap CV vs test (bila kecil -> tidak overfitting):",
      round(top["cv5_accuracy"] - top["test_accuracy"], 4))

        model  varian  cv5_accuracy  cv5_accuracy_std  cv5_f1_macro  cv5_f1_std  test_accuracy  test_f1_macro  best_varian  accuracy_gap
      XGBoost       1        0.6545            0.0122        0.6436      0.0138         0.6384         0.6265          1.0        0.0161
HistGradBoost       1        0.6548            0.0128        0.6428      0.0134         0.6447         0.6337          1.0        0.0101
HistGradBoost       0        0.6546            0.0128        0.6428      0.0137            NaN            NaN          NaN           NaN
HistGradBoost       2        0.6534            0.0110        0.6420      0.0121            NaN            NaN          NaN           NaN
      XGBoost       0        0.6512            0.0109        0.6409      0.0124            NaN            NaN          NaN           NaN
      SVM-RBF       1        0.6429            0.0172        0.6390      0.0181         0.6416         0.6307          1.0        0.0013
 RandomForest       0        0.6448      


MODEL TERBAIK: XGBoost varian 1 | cv5_f1 = 0.6436 | test acc = 0.6384
Gap CV vs test (bila kecil -> tidak overfitting): 0.0161


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Evaluasi memakai model kandidat pemenang yang HANYA dilatih di train (EVAL_FITTED),
# bukan jatim_best.joblib (yang sudah di-refit ke seluruh data untuk deployment).
clf = EVAL_FITTED[res.iloc[0]["model"]]
pv = clf.predict(X[idx_te])
labs = [NAMES[i] for i in sorted(NAMES)]

cm = confusion_matrix(y[idx_te], pv, labels=sorted(NAMES))
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm)
for i in range(6):
    for j in range(6):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=10)
ax.set_xticks(range(6), labs, rotation=20)
ax.set_yticks(range(6), labs)
ax.set_title("Confusion matrix (piksel uji, poligon terpisah)")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(IMG_DIR, "fig_cm_62.png"), dpi=100)
plt.close(fig)

if hasattr(clf, "feature_importances_"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(FEATS, clf.feature_importances_)
    ax.set_title("Feature importance")
    fig.tight_layout()
    fig.savefig(os.path.join(IMG_DIR, "fig_importance_62.png"), dpi=100)
    plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
for cid in sorted(NAMES):
    m_ = df[df.class_id == cid][BANDS].mean().to_numpy()
    ax.plot(BANDS, m_, marker="o", label=f"{cid} {NAMES[cid]}")
ax.set_title("Tanda tangan spektral rata-rata per kelas (pixel, masked)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(IMG_DIR, "fig_spektral_62.png"), dpi=100)
plt.close(fig)
print("tersimpan: fig_cm_62, fig_importance_62, fig_spektral_62")
print(classification_report(y[idx_te], pv, target_names=labs, zero_division=0))

tersimpan: fig_cm_62, fig_importance_62, fig_spektral_62
              precision    recall  f1-score   support

       Sawah       0.42      0.58      0.49      3000
    Bangunan       0.59      0.67      0.63      3000
    Mangrove       0.85      0.79      0.82      2963
 Lahan hijau       0.42      0.31      0.35      2991
        Laut       0.99      0.99      0.99      3000
       Danau       0.55      0.43      0.48      1950

    accuracy                           0.64     16904
   macro avg       0.64      0.63      0.63     16904
weighted avg       0.64      0.64      0.64     16904

